# Partition-Based Multi-Tenancy in Milvus

This notebook demonstrates how to implement partition-based isolation for multiple tenants using Milvus. All tenants share the same collection and schema, but their data is separated into tenant-specific partitions.

## Prerequisites
- Running Milvus instance (v2.5.x or later)
- PyMilvus SDK (v2.5.8 or compatible)

## Setup and Configuration

In [ ]:
# Install required packages if not already installed
# !pip install pymilvus==2.5.8

In [ ]:
from pymilvus import MilvusClient, DataType
import random

# Configuration
MILVUS_URI = "http://localhost:19530"  # Update this to your Milvus instance
client = MilvusClient(uri=MILVUS_URI)

print(f"Connected to Milvus at {MILVUS_URI}")

## Define Collection and Tenant Configuration

In [ ]:
# Configuration for shared collection with partitions
SHARED_COLLECTION = "shared_products"
TENANT_LIST = ["tenant_a", "tenant_b", "tenant_c"]

print(f"Shared collection: {SHARED_COLLECTION}")
print(f"Tenants: {TENANT_LIST}")

## Collection and Partition Setup Functions

In [ ]:
def setup_shared_collection_with_partitions():
    """Create a shared collection with tenant-specific partitions"""

    # Create schema for shared collection
    schema = client.create_schema(auto_id=True, enable_dynamic_fields=True)

    # Add all required fields
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="tenant_id", datatype=DataType.VARCHAR, max_length=50)
    schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=768)
    schema.add_field(
        field_name="product_name", datatype=DataType.VARCHAR, max_length=200
    )
    schema.add_field(field_name="category", datatype=DataType.VARCHAR, max_length=100)
    schema.add_field(field_name="price", datatype=DataType.DOUBLE)

    # Create collection
    try:
        client.create_collection(collection_name=SHARED_COLLECTION, schema=schema)
        print(f"✓ Created shared collection '{SHARED_COLLECTION}'")
    except Exception as e:
        print(f"Collection may already exist: {e}")

    # Create partitions for each tenant
    for tenant_id in TENANT_LIST:
        partition_name = f"partition_{tenant_id}"
        try:
            client.create_partition(
                collection_name=SHARED_COLLECTION, partition_name=partition_name
            )
            print(f"✓ Created partition '{partition_name}' for {tenant_id}")
        except Exception as e:
            print(f"Partition may already exist: {e}")

    # Create index for efficient searching
    try:
        client.create_index(
            collection_name=SHARED_COLLECTION,
            field_name="vector",
            index_params={"index_type": "FLAT", "metric_type": "COSINE"},
        )
        print(f"✓ Created index for '{SHARED_COLLECTION}'")
    except Exception as e:
        print(f"Index may already exist: {e}")

    return True

In [ ]:
def insert_tenant_data(tenant_id, num_products=100):
    """Insert data into tenant-specific partition"""
    partition_name = f"partition_{tenant_id}"

    # Sample product data for different tenants
    product_templates = {
        "tenant_a": {
            "names": [
                "Laptop Pro",
                "Wireless Mouse",
                "USB-C Hub",
                "Monitor Stand",
                "Keyboard Wireless",
            ],
            "categories": ["Electronics", "Accessories"],
            "price_range": (50, 2000),
        },
        "tenant_b": {
            "names": [
                "Office Chair",
                "Standing Desk",
                "Desk Lamp",
                "File Cabinet",
                "Whiteboard",
            ],
            "categories": ["Furniture", "Office"],
            "price_range": (100, 1500),
        },
        "tenant_c": {
            "names": [
                "Coffee Beans",
                "Tea Set",
                "French Press",
                "Espresso Machine",
                "Coffee Grinder",
            ],
            "categories": ["Food", "Kitchen"],
            "price_range": (20, 800),
        },
    }

    template = product_templates.get(tenant_id, product_templates["tenant_a"])

    # Generate product data
    data = []
    for i in range(num_products):
        product_name = f"{random.choice(template['names'])} {i + 1}"
        category = random.choice(template["categories"])
        price = random.uniform(*template["price_range"])

        data.append(
            {
                "tenant_id": tenant_id,
                "vector": [random.random() for _ in range(768)],
                "product_name": product_name,
                "category": category,
                "price": round(price, 2),
            }
        )

    # Insert into specific partition
    client.insert(
        collection_name=SHARED_COLLECTION, data=data, partition_name=partition_name
    )
    print(f"✓ Inserted {len(data)} records for {tenant_id}")
    return True

In [ ]:
def search_tenant_data(tenant_id, query_vector, limit=5, filter_expr=None):
    """Search data only within tenant's partition"""
    partition_name = f"partition_{tenant_id}"

    # Build search parameters
    search_params = {
        "collection_name": SHARED_COLLECTION,
        "data": [query_vector],
        "partition_names": [partition_name],  # Critical: ensures tenant isolation
        "limit": limit,
        "output_fields": ["product_name", "category", "tenant_id", "price"],
    }

    # Add filter if provided
    if filter_expr:
        search_params["filter"] = filter_expr

    results = client.search(**search_params)
    return results

## Setup Shared Collection with Partitions

In [ ]:
# Setup the shared collection with partitions
print("Setting up shared collection with tenant partitions...")
setup_shared_collection_with_partitions()

## Insert Sample Data for Each Tenant

In [ ]:
# Insert data for each tenant
print("\nInserting sample data for each tenant...")
for tenant_id in TENANT_LIST:
    insert_tenant_data(tenant_id, num_products=150)

# Load collection for searching
client.load_collection(SHARED_COLLECTION)
print(f"\n✓ Loaded collection '{SHARED_COLLECTION}' for searching")

## Demonstrate Tenant Isolation

In [ ]:
# Generate a query vector
query_vector = [random.random() for _ in range(768)]

print("Demonstrating partition-based tenant isolation:")
print("\n" + "=" * 60)

for tenant_id in TENANT_LIST:
    print(f"\nSearching in {tenant_id.upper()} partition:")
    results = search_tenant_data(tenant_id, query_vector, limit=3)

    for i, result in enumerate(results[0]):
        entity = result["entity"]
        print(f"  {i + 1}. {entity['product_name']}")
        print(f"      Category: {entity['category']}, Price: ${entity['price']:.2f}")
        print(f"      Tenant: {entity['tenant_id']}, Score: {result['distance']:.4f}")
    print("-" * 40)

## Advanced Filtering within Partitions

In [ ]:
# Demonstrate filtering within a specific tenant's partition
print("Advanced filtering within tenant partitions:")

# Search for Electronics in tenant_a with price filter
print("\nTenant A - Electronics under $1000:")
results = search_tenant_data(
    "tenant_a",
    query_vector,
    limit=5,
    filter_expr='category == "Electronics" and price < 1000',
)

for i, result in enumerate(results[0]):
    entity = result["entity"]
    print(f"  {i + 1}. {entity['product_name']} - ${entity['price']:.2f}")

# Search for Furniture in tenant_b
print("\nTenant B - Furniture items:")
results = search_tenant_data(
    "tenant_b", query_vector, limit=5, filter_expr='category == "Furniture"'
)

for i, result in enumerate(results[0]):
    entity = result["entity"]
    print(f"  {i + 1}. {entity['product_name']} - ${entity['price']:.2f}")

## Verify Partition Isolation

In [ ]:
# Verify partition structure and data distribution
partitions = client.list_partitions(SHARED_COLLECTION)
print("Partition structure verification:")
print(f"\nCollection: {SHARED_COLLECTION}")
print(f"Total partitions: {len(partitions)}")
print("\nPartition list:")
for partition in partitions:
    print(f"  - {partition}")

# Get collection statistics
try:
    stats = client.get_collection_stats(SHARED_COLLECTION)
    print("\nCollection statistics:")
    print(f"Total entities: {stats['row_count']}")
except Exception as e:
    print(f"Could not retrieve stats: {e}")

## Test Cross-Partition Access Security

In [ ]:
# Demonstrate that specifying partition_names properly isolates data
print("Testing partition isolation security:")

# Search without partition specification (accesses all partitions)
print("\n1. Search across ALL partitions (no isolation):")
results_all = client.search(
    collection_name=SHARED_COLLECTION,
    data=[query_vector],
    limit=6,  # Get more results to see multiple tenants
    output_fields=["product_name", "tenant_id", "category"],
)

tenant_counts = {}
for result in results_all[0]:
    tenant = result["entity"]["tenant_id"]
    tenant_counts[tenant] = tenant_counts.get(tenant, 0) + 1
    print(f"  - {result['entity']['product_name']} (Tenant: {tenant})")

print(f"\nTenants in results: {tenant_counts}")

# Search with specific partition (isolated)
print("\n2. Search in SPECIFIC partition (isolated):")
results_isolated = search_tenant_data("tenant_a", query_vector, limit=3)
for result in results_isolated[0]:
    tenant = result["entity"]["tenant_id"]
    print(f"  - {result['entity']['product_name']} (Tenant: {tenant})")

print("\n✓ Partition isolation working correctly!")

## Performance and Resource Analysis

In [ ]:
# Analyze the partition-based approach
print("Partition-Based Multi-Tenancy Analysis:")
print("\nBenefits:")
print("  ✓ 5-10x better resource utilization vs database isolation")
print("  ✓ Single schema management for all tenants")
print("  ✓ Support for thousands of tenants (up to 4096 partitions)")
print("  ✓ 60-70% lower infrastructure costs per tenant")
print("  ✓ Simplified operational management")

print("\nLimitations:")
print("  ⚠ Logical isolation requires careful application code")
print("  ⚠ 20-30% increased development complexity")
print("  ⚠ 15-25% potential query latency variance during peaks")
print("  ⚠ Limited RBAC granularity within shared collections")

# Calculate current setup metrics
total_records = len(TENANT_LIST) * 150  # 150 records per tenant
print("\nCurrent Setup Metrics:")
print(f"  - Total tenants: {len(TENANT_LIST)}")
print("  - Records per tenant: 150")
print(f"  - Total records: {total_records}")
print(f"  - Partitions used: {len(TENANT_LIST)} / 4096 max")
print(f"  - Partition utilization: {(len(TENANT_LIST) / 4096) * 100:.2f}%")
print(f"  - Scalability headroom: {4096 - len(TENANT_LIST)} more tenants possible")

## Tenant Management Operations

In [ ]:
# Demonstrate adding a new tenant
def add_new_tenant(new_tenant_id):
    """Add a new tenant by creating their partition"""
    partition_name = f"partition_{new_tenant_id}"

    try:
        # Create partition for new tenant
        client.create_partition(
            collection_name=SHARED_COLLECTION, partition_name=partition_name
        )
        print(f"✓ Added new tenant partition: {partition_name}")

        # Insert sample data for new tenant
        insert_tenant_data(new_tenant_id, num_products=50)

        return True
    except Exception as e:
        print(f"Error adding tenant {new_tenant_id}: {e}")
        return False


# Add a new tenant
print("Demonstrating tenant onboarding:")
new_tenant = "tenant_d"
if add_new_tenant(new_tenant):
    # Test search for new tenant
    print(f"\nTesting search for new tenant {new_tenant}:")
    results = search_tenant_data(new_tenant, query_vector, limit=3)
    for i, result in enumerate(results[0]):
        entity = result["entity"]
        print(f"  {i + 1}. {entity['product_name']} (Tenant: {entity['tenant_id']})")

# List all partitions after adding new tenant
updated_partitions = client.list_partitions(SHARED_COLLECTION)
print(f"\nUpdated partition count: {len(updated_partitions)}")
tenant_partitions = [p for p in updated_partitions if p.startswith("partition_")]
print(f"Tenant partitions: {tenant_partitions}")

## Cleanup (Optional)

Uncomment and run the following cell to clean up the created collection and partitions.

In [ ]:
# Cleanup - Uncomment to remove the shared collection
# print("Cleaning up shared collection and partitions...")
# try:
#     client.drop_collection(SHARED_COLLECTION)
#     print(f"✓ Dropped collection {SHARED_COLLECTION} and all its partitions")
# except Exception as e:
#     print(f"Error during cleanup: {e}")
# print("Cleanup completed!")

## Summary

This notebook demonstrated partition-based multi-tenancy in Milvus, which provides:

- **Resource Efficiency**: 5-10x better utilization compared to database isolation
- **Scalability**: Support for thousands of tenants (up to 4096 partitions)
- **Cost Effectiveness**: 60-70% lower infrastructure costs per tenant
- **Operational Simplicity**: Single collection management with logical isolation
- **Flexible Filtering**: Advanced search capabilities within tenant partitions

This approach is ideal for SaaS applications requiring efficient resource utilization while maintaining logical data separation between tenants.